In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [15]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

import os, json, time, hmac, hashlib, requests
#from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
#load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
# 어제 했던것, API 및 HTTP 이용함
# 페이지네이션 해서 가져오는걸 할 예정
# https://jsonplaceholder.typicode.com/posts
  # 천만건 요청하면 천만건 모두 응답하지 않을 수 있음
  # 응답 속도 등..
# 페이지네이션 이용해서 조금씩만 가져올 수 있도록
# 아래와같이 페이지 관련 파라미터
  # 한번에 가져갈 수 있는 데이터의 리밋을 지정
  # https://jsonplaceholder.typicode.com/posts/?_page=1&_limit=10


In [5]:
BASE_URL = "https://jsonplaceholder.typicode.com"

In [21]:
# 한번에 10개, 최대 100개 등
def fetch_all_posts(page_size = 10, max_pages = 100):
  page = 1
  while page <= max_pages:
    r = requests.get(f"{BASE_URL}/posts", params={"_page": page, "_limit": page_size})
    items = r.json()

    if not items:
      break # 아이템이 없다면 break
    print(f"page {page} : {len(items)}")
    for item in items:
      yield item

    if len(items) < page_size:
      break

    page += 1

In [22]:
all_posts = list(fetch_all_posts(page_size = 20))

page 1 : 20
page 2 : 20
page 3 : 20
page 4 : 20
page 5 : 20


In [ ]:
all_posts

In [20]:
len(all_posts), all_posts[0]['title']

(10,
 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit')

In [27]:
def cursor_painate_simulator(total_items=25, page_size=10):
  all_data = [{'id' : i, "name" : f"item-{i}"} for i in range(1, total_items+1)]

  def fetch_page(cursor = None, limit = page_size):
    start = int(cursor) if cursor else 0
    end = min(start + limit, total_items)
    items = all_data[start:end]
    next_cursor = str(end) if end < total_items else None

    return {'items' : items, 'next_cursor': next_cursor}


  return fetch_page

fetch = cursor_painate_simulator(total_items = 25, page_size = 10)

# 함수르 실행시키면 fetch_page이 반환되는데,
  # cursor와 limit이 적절하게 채워진 해당 함수가 반환될 것


In [28]:
cursor = None
page = 1
all_items = []
while True:
  result = fetch(cursor=cursor) # result = {'items' : items, 'next_cursor': next_cursor}
  print(f"page : {page} | {len(result['items'])}, next = {result['next_cursor']}")
  all_items.extend(result['items']) # append는 list의 요소를 추가하는것임, 두개의 리스트를 이을때는 extend임

  if not result['next_cursor']:
    break
  curosr = result['next_cursor']
  page += 1



스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
page : 1760508 | 10, next = 10
page : 1760509 | 10, next = 10
page : 1760510 | 10, next = 10
page : 1760511 | 10, next = 10
page : 1760512 | 10, next = 10
page : 1760513 | 10, next = 10
page : 1760514 | 10, next = 10
page : 1760515 | 10, next = 10
page : 1760516 | 10, next = 10
page : 1760517 | 10, next = 10
page : 1760518 | 10, next = 10
page : 1760519 | 10, next = 10
page : 1760520 | 10, next = 10
page : 1760521 | 10, next = 10
page : 1760522 | 10, next = 10
page : 1760523 | 10, next = 10
page : 1760524 | 10, next = 10
page : 1760525 | 10, next = 10
page : 1760526 | 10, next = 10
page : 1760527 | 10, next = 10
page : 1760528 | 10, next = 10
page : 1760529 | 10, next = 10
page : 1760530 | 10, next = 10
page : 1760531 | 10, next = 10
page : 1760532 | 10, next = 10
page : 1760533 | 10, next = 10
page : 1760534 | 10, next = 10
page : 1760535 | 10, next = 10
page : 1760536 | 10, next = 10
page : 1760537 | 10, next = 10
page : 1760538 | 10, next = 10
pag

KeyboardInterrupt: 

In [29]:
all_items

[{'id': 1, 'name': 'item-1'},
 {'id': 2, 'name': 'item-2'},
 {'id': 3, 'name': 'item-3'},
 {'id': 4, 'name': 'item-4'},
 {'id': 5, 'name': 'item-5'},
 {'id': 6, 'name': 'item-6'},
 {'id': 7, 'name': 'item-7'},
 {'id': 8, 'name': 'item-8'},
 {'id': 9, 'name': 'item-9'},
 {'id': 10, 'name': 'item-10'},
 {'id': 1, 'name': 'item-1'},
 {'id': 2, 'name': 'item-2'},
 {'id': 3, 'name': 'item-3'},
 {'id': 4, 'name': 'item-4'},
 {'id': 5, 'name': 'item-5'},
 {'id': 6, 'name': 'item-6'},
 {'id': 7, 'name': 'item-7'},
 {'id': 8, 'name': 'item-8'},
 {'id': 9, 'name': 'item-9'},
 {'id': 10, 'name': 'item-10'},
 {'id': 1, 'name': 'item-1'},
 {'id': 2, 'name': 'item-2'},
 {'id': 3, 'name': 'item-3'},
 {'id': 4, 'name': 'item-4'},
 {'id': 5, 'name': 'item-5'},
 {'id': 6, 'name': 'item-6'},
 {'id': 7, 'name': 'item-7'},
 {'id': 8, 'name': 'item-8'},
 {'id': 9, 'name': 'item-9'},
 {'id': 10, 'name': 'item-10'},
 {'id': 1, 'name': 'item-1'},
 {'id': 2, 'name': 'item-2'},
 {'id': 3, 'name': 'item-3'},
 {'i

In [ ]:
# 실습 : 페이지 별로 가져올거임
# todos 페이지 : https://jsonplaceholder.typicode.com/todos

# 1. completed = True 인 todo만 가져옴
# 2. 20개가 모이면 중단
# 3. completed = True인 todo list 20개를 리턴

def collect_completed_todos(max_items = 20, page_size = 10):
  page = 1
  collected_items = 0
  while collected_items < max_items:
    r = requests.get(f"{BASE_URL}/todos", params={"_page": page, "_limit": page_size})
    items = r.json()

    if not items:
      break
    print(f"page {page} : {len(items)}")

  return

1. 로직의 흐름 (Step-by-Step)
상자(Page) 요청: 서버라는 창고에 가서 "1번 상자 주세요. (한 상자엔 사과 10개가 들어있어요)"라고 요청합니다. (page_size=10)
내용물 확인 (Iteration): 상자를 열어 사과 10개를 하나씩 꺼내봅니다. (for item in items)
조건 검사 (Filtering): 사과가 **빨간색(completed: True)**인지 확인합니다.
빨간색이면? 내 **바구니(results)**에 담습니다.
빨간색이 아니면? 그냥 버립니다(무시합니다).
개수 체크: 내 바구니에 사과가 **20개(max_items)**가 찼는지 확인합니다.
20개가 됐다면? 뒤에 남은 사과가 더 있든 말든, 다음 상자가 있든 말든 즉시 작업을 멈추고 집에 갑니다. (return results)
아직 부족하다면? 다음 상자(2번 상자, 3번 상자...)를 달라고 해서 2번 과정부터 다시 반복합니다. (page += 1)

In [33]:
def collect_completed_todos(max_items=20, page_size=10):
    collected = []
    page = 1

    while len(collected) < max_items:
        r = requests.get(f"{BASE_URL}/todos", params={"_page": page, "_limit": page_size})
        items = r.json()

        if not items:
            break

        print(f"page {page} : {len(items)}")

        for item in items:
            if item.get('completed') == True:
                collected.append(item)

                if len(collected) == max_items:
                    print(f"max_items: {max_items}")
                    return collected
        page += 1

    return collected[:max_items]

In [34]:
completed_list = collect_completed_todos(20, 10)

page 1 : 10
page 2 : 10
page 3 : 10
page 4 : 10
page 5 : 10
max_items: 20


In [42]:
def get_by_path(data, path, default=None):
  current = data
  for key in path.split("."): # [address, city]
    if isinstance(current, dict) and key in current:
      current = current[key]
    else:
      return default
  return current


user_data = requests.get(f"{BASE_URL}/users/1").json()
user_data
get_by_path(user_data, 'name')

'Leanne Graham'

In [43]:
# 데이터 가져와서 마크다운 형식으로 변환

# 헬퍼함수 get_by_path
def get_by_path(data, path, default=None):
  current = data
  for key in path.split("."): # [address, city]
    if isinstance(current, dict) and key in current:
      current = current[key]
    else:
      return default
  return current

# 본 함수
def users_to_markdown(user_ids : list, fields : list):
  rows = []
  for uid in user_ids:
    r = requests.get(f"{BASE_URL}/users/{uid}") # https://jsonplaceholder.typicode.com/users/1
    if not r.ok:
      continue
    data = r.json()
    row = {f:get_by_path(data, f, "N/A") for f in fields} # get_by_path 함수는 속성안에 또다른 속성이 있는 경우를 가져올 떄
    rows.append(row)

  header = "| " + " | ".join(fields) + " |"  # **abc** 볼드체.. string으로 되는 등 주의
  # | | name | email | company |
  sep = "|" + "|".join(["-"*(len(f) + 2) for f in fields]) + "|"
  body_lines = []
  for row in rows:
    line = "| " + " | ".join(str(row[f]) for f in fields) + " |"
    body_lines.append(line)

  return '\n'.join([header, sep] + body_lines)

table = users_to_markdown(user_ids = [1,2,3], fields = ['name', 'email', 'address.city'])
# | abc | def | ... 마크다운 표

In [45]:
print(table)

| name | email | address.city |
|------|-------|--------------|
| Leanne Graham | Sincere@april.biz | Gwenborough |
| Ervin Howell | Shanna@melissa.tv | Wisokyburgh |
| Clementine Bauch | Nathan@yesenia.net | McKenziehaven |


In [ ]:
# userid에는 [1,2,3,...]
# fields에는 ['name', 'email', 'company'...]
# 경로의 users 예시
# {
#   "id": 1,
#   "name": "Leanne Graham",
#   "username": "Bret",
#   "email": "Sincere@april.biz",
#   "address": {
#     "street": "Kulas Light",
#     "suite": "Apt. 556",
#     "city": "Gwenborough",
#     "zipcode": "92998-3874",
#     "geo": {
#       "lat": "-37.3159",
#       "lng": "81.1496"
#     }
#   },
#   "phone": "1-770-736-8031 x56442",
#   "website": "hildegard.org",
#   "company": {
#     "name": "Romaguera-Crona",
#     "catchPhrase": "Multi-layered client-server neural-net",
#     "bs": "harness real-time e-markets"
#   }
# }

### API(LLM 붙이기 전

In [ ]:
# https://jsonplaceholder.typicode.com/users/1 유저 1번 정보는 여기
# https://jsonplaceholder.typicode.com/post?userId=1 # 유저 1번이 쓴 포스트는 여기있음


In [51]:
def build_user_profile(user_id):
  # 기본정보, 게시글, Todo 등을 순차적으로 뽑을 것
  profile = {'user_id' : user_id}

  # 기본정보
  try:
    r = requests.get(f"{BASE_URL}/users/{user_id}")
    if r.ok:
      u = r.json()
      profile['name'] = u['name']
      profile['email'] = u['email']
      profile['city'] = u['address']['city']

  except:
    profile['user_error'] = 'error'

  # 포스트
  try:
    r = requests.get(f"{BASE_URL}/posts", params = {'userId': user_id})
    if r.ok:
      posts = r.json()
      profile['post_count'] = len(posts)
      profile['recent_posts'] = [p['title'][:40] for p in posts[:3]]
  except:
    profile['post_error'] = 'error'

  #
  try:
    r = requests.get(f"{BASE_URL}/todos", params = {'userId' : user_id})
    if r.ok:
      todos = r.json()
      profile['todo_total'] = len(todos)
      profile['todo_completed'] = sum(1 for t in todos if t['completed'])
  except:
    profile['todo_error'] = 'error'

  return profile

In [52]:
build_user_profile(1)

{'user_id': 1,
 'name': 'Leanne Graham',
 'email': 'Sincere@april.biz',
 'city': 'Gwenborough',
 'post_count': 10,
 'recent_posts': ['sunt aut facere repellat provident occae',
  'qui est esse',
  'ea molestias quasi exercitationem repell'],
 'todo_total': 20,
 'todo_completed': 11}

In [ ]:
# 실습, 아래 결과를 내는 함수 생성
build_team_report([1,2,3,4,5])
{
 'team_size':
 "members" :[
     {'user_id':, 'name':, 'post_count':}
 ]
 'top_poster' : {'user_id': user_id}, "post_count": int}

}

In [53]:
def build_team_report(user_list):
    post_r = requests.get(f"{BASE_URL}/posts")
    all_posts = post_r.json()

    post_counts = {}
    for post in all_posts:
        uid = post['userId']
        post_counts[uid] = post_counts.get(uid, 0) + 1

    report = {
        'team_size': len(user_list),
        'members': [],
        'top_poster': {'user_id': None, 'post_count': -1}
    }
    for user_id in user_list:
        user_r = requests.get(f"{BASE_URL}/users/{user_id}")
        u = user_r.json()

        current_count = post_counts.get(user_id, 0)

        member_data = {
            'user_id': user_id,
            'name': u.get('name'),
            'post_count': current_count
        }
        report['members'].append(member_data)

        if current_count > report['top_poster']['post_count']:
            report['top_poster'] = {
                'user_id': user_id,
                'post_count': current_count
            }

    return report


In [56]:
# 강사님 코드
def build_team_report(user_list):
  members = []
  for uid in user_list:
    try:
      u = requests.get(f"{BASE_URL}/users/{uid}")
      p = requests.get(f"{BASE_URL}/posts", params = {'userId':uid})
      t = requests.get(f"{BASE_URL}/todos", params = {'userId':uid})

      if not (u.ok and p.ok and t.ko):
        continue
      todos = t.json()
      members.append({
          'user_id' :uid,
          'name' : u.json()['name'],
          'post_count' : len(p.json())
      })
    except:
      continue

  total_posts = sum(m['post_count'] for m in members)
  top = max(members, key =lambda m : m['post_count']) if members else None

  return{
      'team_size' : len(members),
      'members' : members,
      'top_poster' : {'user_id': top['user_id', 'post_count' : top['post_count']]} if top else None

  }

In [57]:
build_team_report([1,2,3,4,5])

{'team_size': 0, 'members': [], 'top_poster': None}

### 오케스트레이터
- 하네스 엔지니어링

In [ ]:
# #@tool
# def abc:
#   return ~~

# llm_with_tool = llm.bind_tools([abc, abc2, abc3, ...])
# llm_with_tool.invoke("~~~ 찾아줘")

# llm_with_tool 이라는 애가, 알아서 어떤 툴을 자동으로 찾고
# 파라미터를 넘기고, 결과를 이용해서 연속된 다음 툴들을 호출하여 결과를 반환함
# 추가로 예외처리, 무한루프 처리 등을 추가 --> 오케스트레이터

In [58]:
@tool
def api_get_user(user_id):
  """사용자 기본 정보(이름, 이메일, 도시)를 조회합니다."""
  r = requests.get(f"{BASE_URL}/users/{user_id}")
  u = r.json()
  return json.dumps({'name': u['name'], 'email': u['email'], 'city': u['address']['city']}, ensure_ascii=False)

@tool
def api_get_user_posts(user_id, limit=5):
  """사용자의 최근 게시글 제목을 조회합니다."""
  r = requests.get(f"{BASE_URL}/posts", params = {'user_Id': user_id})
  posts = r.json()[:limit]
  return json.dumps([{'id': p['id'], 'title': p['title']} for p in posts], ensure_ascii= False)

@tool
def api_get_user_todos(user_id):
  """ 사용자의 TODO 완료율을 조회합니다."""
  r = requests.get(f"{BASE_URL}/todos", params = {'user_Id': user_id})
  todos = r.json()
  done = sum(1 for t in todos if t['completed'])
  return json.dumps({'completed': done, 'total': len(todos), 'rate' : f"{done/len(todos)*100:0f}%"}, ensure_ascii=False)

api_tools = [api_get_user, api_get_user_posts, api_get_user_todos]
tool_map = {t.name: t for t in api_tools}

print(f"registered tools: {[t.name for t in api_tools]}")

registered tools: ['api_get_user', 'api_get_user_posts', 'api_get_user_todos']


In [61]:
# 오케스트레이터
  # 중간중간의 에러 처리, 방지 등등
def orchestrate(query, max_turns = 5):
  llm_api = llm.bind_tools(api_tools)
  messages = [HumanMessage(content = query)]

  tool_call_count = 0
  MAX_TOOL_CALLS = 10 # 툴끼리 핑퐁하는 경우를 방지

  for turn in range(max_turns):
    try:
      resp = llm_api.invoke(messages)
    except Exception as e:
      return f"llm call fail {e}"

    messages.append(resp)

    if not resp.tool_calls:
      return resp.content

    for tc in resp.tool_calls: # 툴을 잘못 가져오는 경우도 있음
      name = tc['name']
      args = tc['args']

      if name not in tool_map:
        result = f"Error : unknown tool '{name}' "
      else:
        tool_call_count += 1
        if tool_call_count > MAX_TOOL_CALLS:
          return f"Error : tool call limit exceeded"

        try:
          result = tool_map[tc['name']].invoke(tc['args'])
        except:
          result = f"Error"

      print(f"[Trun {turn+1}] {tc['name']} ({tc['args']}) -> {result[:80]}")
      messages.append(ToolMessage(content=result, tool_call_id = tc['id']))

  return "최대 턴 초과"

orchestrate('1번 유저의 정보와 최근 게시글 3개, todo 완료율을 알려줘')

[Trun 1] api_get_user ({'user_id': 1}) -> {"name": "Leanne Graham", "email": "Sincere@april.biz", "city": "Gwenborough"}
[Trun 1] api_get_user_posts ({'user_id': 1, 'limit': 3}) -> [{"id": 1, "title": "sunt aut facere repellat provident occaecati excepturi opti
[Trun 1] api_get_user_todos ({'user_id': 1}) -> {"completed": 90, "total": 200, "rate": "45.000000%"}


'1번 유저의 정보는 다음과 같습니다:\n\n- **이름**: Leanne Graham\n- **이메일**: Sincere@april.biz\n- **도시**: Gwenborough\n\n최근 게시글 3개는 다음과 같습니다:\n\n1. **sunt aut facere repellat provident occaecati excepturi optio reprehenderit**\n2. **qui est esse**\n3. **ea molestias quasi exercitationem repellat qui ipsa sit aut**\n\nTODO 완료율은 다음과 같습니다:\n\n- **완료된 항목**: 90\n- **총 항목**: 200\n- **완료율**: 45.00%'

### 커스텀 툴

In [62]:
from langchain_core.tools import tool, StructuredTool, BaseTool
## @tool
# def discuss(a_message,b_message):
#   """a b의 토론 결과를 리턴합니다."""
#   a = []
#   b = []

#   return result

# 툴의 경우 llm invoke로 호출할때, a, b 및 return만 중요함
  # 중간의 a의 대화, b의 대화 등 중간 결과는 볼수가 없음

In [63]:
# BaseTool 활용
# 인스턴스 변수, 캐시 ,,,

class APICallerTool(BaseTool): # BaseTool의 경우 invoke할 떄 사용됨
  name : str = 'api_caller'
  description : str = "REST API를 호출하여 데이터를 조회합니다."
  call_count : int = 0
  cache : dict = {}

  def _run(self, endpoint: str) -> str:
    # 캐시 확인
    if endpoint in self.cache:
      return f"[cache] {self.cache[endpoint]}"

    self.call_count += 1
    data = {
        'users/1' : {'name' : 'abcd', 'email' : 'abcd@example.com'},
        'posts/1' : {'title' : '1st post', 'userId' : 1},
        'products/1' : {'name': 'labtop', 'price' : 150000}
    }

    result = json.dumps(data.get(endpoint, {'error' : 'not found'}), ensure_ascii=False)

    self.cache[endpoint] = result
    return f"[API #{self.call_count}] {result}"

In [64]:
api_tool = APICallerTool()
api_tool.invoke({'endpoint': 'users/1'})

'[API #1] {"name": "abcd", "email": "abcd@example.com"}'

In [65]:
## @tool : stateless 캐시, 버전, 히스토리 등을 state라고 하는데 이게 없다..

# pydantic 이용..

In [85]:
from typing import Type
class StockInput(BaseModel): #BaseModel은 pydantic과 관련된 클래스
  ticker : str = Field(description = '주식 티커 (예: AAPL)')
  include_history : bool = Field(default= False, description = "과거 데이터 포함 여부")

class StockTool(BaseTool):
  name : str = 'stock_lookup'
  description : str = '주식 정보를 조회합니다.'
  args_schema : Type[BaseModel] = StockInput # BaseModel는 데이터 스키마를 정의하는 클래스임
  #query_log : list = []
  query_log: list = Field(default_factory=list)

  def _run(self, ticker:str, include_history: bool = False) -> str:
    prices = {"AAPL" : 100.5, "GOOGL": 102.1, "TSLA": 280.3}
    price = prices.get(ticker.upper())
    if not price:
      return f"{ticker} no data"

    self.query_log.append({'ticker': ticker, 'time': time.time()})
    result = f"{ticker} : ${price}"

    if include_history:
      result += f"  (30일 변동 : + {price*0.05})"

    return result

In [88]:
stock = StockTool()

In [89]:
stock.invoke({'ticker': 'AAPL'})

'AAPL : $100.5'

In [90]:
stock.invoke({'ticker': 'GOOGL', 'include_history': True})

'GOOGL : $102.1  (30일 변동 : + 5.105)'

In [81]:
stock.query_log

[{'ticker': 'AAPL', 'time': 1777036023.2285104},
 {'ticker': 'AAPL', 'time': 1777036030.3721673},
 {'ticker': 'GOOGL', 'time': 1777036048.1489768},
 {'ticker': 'AAPL', 'time': 1777036065.9665756},
 {'ticker': 'GOOGL', 'time': 1777036066.6240363},
 {'ticker': 'AAPL', 'time': 1777036069.071216},
 {'ticker': 'GOOGL', 'time': 1777036069.7831023},
 {'ticker': 'GOOGL', 'time': 1777036095.8631952}]

In [ ]:
# 검색결과를 캐싱해주는 클래스
# 검색 요청이 들어오면
#   ├─ 이전에 검색한 적 있음 (캐시 HIT) → 저장된 결과 바로 반환
#   └─ 처음 검색 (캐시 MISS) → API 호출 후 결과를 캐시에 저장 → 반환

# class CachedSearchTool:

#   _run(self, query) -> str:
#   query가 self.cache에 존재하면 -> 캐시에 있는 결과 리턴
#                         없다면 api_call += 1 한 뒤에
#                         query 검색결과 {api_call} 건 -> 캐시에 저장 -> [api] 리턴

# invoke({'query': '파이썬 기초'}) -> 파이썬 기초 api 리턴 -> cache[파이썬 기초]

# 즉 검색 결과를 캐시하는 함수

In [99]:
class CachedSearchTool(BaseTool): # BaseTool 상속!
  name : str = 'cached_search'
  description : str = '캐싱 기능이 있는 검색 툴'
  cache : dict = {}
  api_calls : int = 0

  def _run(self, query:str) -> str:
    if query in self.cache:
      return f"[cache] {self.cache[query]}"

    self.api_calls += 1
    result = f"'{query}' 검색 결과 {self.api_calls} 건"

    self.cache[query] = result
    return f"[API] {result}"


In [100]:
search = CachedSearchTool()
print(search.invoke({'query' : '파이썬 기초'}))
print(search.invoke({'query' : '파이썬 기초'}))

[API] '파이썬 기초' 검색 결과 1 건
[cache] '파이썬 기초' 검색 결과 1 건
